# 365 Probabilidades — Dia #032
## Qual a probabilidade de o hábito da leitura te dar mais de um ano de vida?

**Tipo:** Longevidade / Hábito
**Data de publicação:** 2026-07-15
**Ferramenta:** Python
**Decisão analisada:** Vale a pena reservar 30 minutos por dia pra ler um livro?
**Hashtag:** #365Probabilidades #Dia032

---

### 📖 A História

Uma das coisas boas de sair do mercado financeiro foi reencontrar a leitura.

Enquanto eu estava lá dentro, meu tempo com livros virou aquilo que muita gente conhece: página e meia antes de dormir, olho pesado, o mesmo parágrafo relido três vezes, sono. Não era leitura, era rendição.

Quando saí, o tempo voltou. E os livros também.

E não por acaso, os que me pegaram nesse período foram todos, de algum jeito, sobre reconstruir. **Antifrágil**, do Taleb, sobre o que se fortalece com desordem. **Nunca é Hora de Parar**, do David Goggins, sobre o limite mental antes do físico.

E, num momento muito específico, logo depois que meu pai partiu, li **A Morte: Um Dia que Vale a Pena Viver**, do Rubem Alves. Um livro que se chama sobre morte e é, na verdade, um dos livros mais afiados sobre viver que eu já encontrei.

Meu pai é um dos motores desse projeto. Ele acreditava em aprender todos os dias, e essa é uma das poucas coisas que eu carrego dele que ninguém pode tirar.

Um dia desses, meio por acaso, tropecei num paper de Yale sobre leitura e longevidade. E o número me parou.

---

### 📚 O Conceito: O Livro Como Intervenção Mensurável

Existe uma diferença entre "ler faz bem" (o que todo mundo diz) e "ler prolonga a vida em uma quantidade específica, medida em anos" (o que os epidemiologistas conseguem provar).

A ciência da longevidade tem uma tradição de medir hábitos aparentemente banais e mostrar que eles têm impacto quantificável na sobrevida. Exercício. Vínculo social. Sono. E, mais recentemente, leitura.

O que faz o estudo de Yale ser especial é a combinação de três coisas: amostra grande, acompanhamento longo, e ajuste rigoroso pra tudo que poderia estar confundindo o efeito (riqueza, escolaridade, saúde inicial, depressão, estado civil).

O que eles queriam descobrir era simples: pessoas que leem livros vivem mais que pessoas que não leem? E, se sim, quanto mais?

A resposta foi mais forte do que qualquer um esperava.

---

### 🧮 O Modelo

**Fonte principal:**
- Bavishi, A., Slade, M. D. & Levy, B. R. (2016) — "A chapter a day: Association of book reading with longevity", *Social Science & Medicine*, 164, 44-48 — **N=3.635**, coorte prospectiva do Health and Retirement Study, **12 anos de acompanhamento**, análise Cox proportional hazards com ajuste completo (idade, sexo, raça, escolaridade, comorbidades, saúde autorreferida, riqueza, estado civil, depressão)

**Fontes de reforço:**
- Bygren, Konlaan & Johansson (1996) — *BMJ* — pioneiro sueco sobre atividades culturais e sobrevida
- Chang, Wu & Hsiung (2021) — leitura frequente reduz risco de declínio cognitivo em idosos, com dose-resposta

**Nota metodológica:** neste dia **aplico o fator de correção ×0.80 sobre o "23 meses"**, transformando-o em **~18 meses**. Motivo: os 23 meses derivam de análise sobre padrão de leitura autorrelatado. O modelo estatístico é sólido, mas todo autorrelato tem viés (as pessoas exageram hábitos que consideram virtuosos). Aplicar o fator conservador mantém a mensagem — o efeito é forte — sem repetir o número mais otimista possível.

Os *hazard ratios* originais (HR=0,80, redução de 20% no risco de mortalidade) são preservados, porque são medidas de risco relativo derivadas diretamente do modelo Cox, não estimativas de survey.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("✅ Bibliotecas carregadas")


✅ Bibliotecas carregadas


In [2]:
# --- DADOS DA LITERATURA ---
# Bavishi, Slade & Levy (2016), Social Science & Medicine 164:44-48
# Health and Retirement Study — coorte nacionalmente representativa (EUA)

n_coorte = 3635
anos_seguimento = 12
idade_minima = 50

# Hazard Ratios do modelo Cox (ajustado por: idade, sexo, raça, escolaridade,
# comorbidades, saúde autorreferida, riqueza, estado civil, depressão)
hr_leitores = 0.80              # 20% de redução no risco de mortalidade
p_valor_hr = 0.01

# Efeito dose-resposta (por tercis de horas de leitura semanal)
hr_tercil_2 = 0.83              # leitura moderada (~até 3,5h/semana ~ 30min/dia)
hr_tercil_3 = 0.77              # leitura alta (>3,5h/semana)

# Mortalidade absoluta durante o seguimento
p_morte_nao_leitores = 0.33     # 33% morreram (não leitores)
p_morte_leitores = 0.27         # 27% morreram (leitores)

# Vantagem em meses de sobrevida no ponto de 80% de sobrevivência
meses_vantagem_bruto = 23       # estimativa bruta do modelo não ajustado

# APLICO o fator de correção do projeto (autorrelato)
fator_correcao = 0.80
meses_vantagem_corrigido = meses_vantagem_bruto * fator_correcao

print("=" * 68)
print("  DADOS — LEITURA E LONGEVIDADE (Bavishi, Slade & Levy, 2016)")
print("=" * 68)
print(f"\n  Coorte:              {n_coorte:,} adultos (EUA, 50+)")
print(f"  Seguimento:          {anos_seguimento} anos")
print(f"  Método:              Cox proportional hazards")
print(f"\n  MORTALIDADE NO PERÍODO:")
print(f"  → Não leitores:      {p_morte_nao_leitores*100:.0f}% morreram")
print(f"  → Leitores:          {p_morte_leitores*100:.0f}% morreram")
print(f"\n  HAZARD RATIOS (ajustados por todos os covariáveis):")
print(f"  → Leitores em geral: HR = {hr_leitores}   (redução de {(1-hr_leitores)*100:.0f}%)")
print(f"  → Leitura moderada:  HR = {hr_tercil_2}   (~30min/dia)")
print(f"  → Leitura alta:      HR = {hr_tercil_3}")
print(f"\n  VANTAGEM DE SOBREVIDA:")
print(f"  → Estimativa bruta:              {meses_vantagem_bruto} meses")
print(f"  → Com fator de correção ×0.80:   ~{meses_vantagem_corrigido:.0f} meses")
print(f"                                    (~{meses_vantagem_corrigido/12:.1f} anos)")
print("=" * 68)


  DADOS — LEITURA E LONGEVIDADE (Bavishi, Slade & Levy, 2016)

  Coorte:              3,635 adultos (EUA, 50+)
  Seguimento:          12 anos
  Método:              Cox proportional hazards

  MORTALIDADE NO PERÍODO:
  → Não leitores:      33% morreram
  → Leitores:          27% morreram

  HAZARD RATIOS (ajustados por todos os covariáveis):
  → Leitores em geral: HR = 0.8   (redução de 20%)
  → Leitura moderada:  HR = 0.83   (~30min/dia)
  → Leitura alta:      HR = 0.77

  VANTAGEM DE SOBREVIDA:
  → Estimativa bruta:              23 meses
  → Com fator de correção ×0.80:   ~18 meses
                                    (~1.5 anos)


In [3]:
# --- O MODELO ---
# O retorno por hora investida — traduzindo o benefício em números concretos

# Um "capítulo por dia" ≈ 30 minutos
minutos_por_dia = 30
dias_por_ano = 365
horas_por_ano = minutos_por_dia * dias_por_ano / 60   # 182,5 horas
horas_por_decada = horas_por_ano * 10                  # 1.825 horas

# Retorno em horas de vida a mais (aproximado, usando o valor corrigido)
horas_de_vida_a_mais = meses_vantagem_corrigido * 30 * 24    # meses × dias × horas
retorno_por_hora = horas_de_vida_a_mais / horas_por_decada   # razão

print("=" * 68)
print("  MODELO — O RETORNO DA HORA INVESTIDA")
print("=" * 68)
print(f"\n  30 minutos de leitura por dia:")
print(f"  → {horas_por_ano:.0f} horas por ano")
print(f"  → {horas_por_decada:.0f} horas ao longo de uma década")
print(f"\n  Retorno estimado (com correção ×0.80):")
print(f"  → {meses_vantagem_corrigido:.0f} meses de vida a mais")
print(f"  → ≈ {horas_de_vida_a_mais:,.0f} horas de vida a mais")
print(f"\n  RAZÃO DE RETORNO:")
print(f"  → Cada 1 hora lida ao longo da década")
print(f"    'compra' aproximadamente {retorno_por_hora:.1f} horas de vida a mais.")
print(f"\n  Não conheço muitos hábitos com esse tipo de retorno.")
print("=" * 68)
print(f"\n  ⚠️  NOTA IMPORTANTE:")
print(f"  → Estudo observacional. Mostra ASSOCIAÇÃO, não causalidade direta.")
print(f"  → Modelo ajustou por 9 covariáveis (riqueza, educação, saúde etc.).")
print(f"  → Efeito mediado por escore cognitivo — o cérebro que lê envelhece diferente.")
print("=" * 68)


  MODELO — O RETORNO DA HORA INVESTIDA

  30 minutos de leitura por dia:
  → 182 horas por ano
  → 1825 horas ao longo de uma década

  Retorno estimado (com correção ×0.80):
  → 18 meses de vida a mais
  → ≈ 13,248 horas de vida a mais

  RAZÃO DE RETORNO:
  → Cada 1 hora lida ao longo da década
    'compra' aproximadamente 7.3 horas de vida a mais.

  Não conheço muitos hábitos com esse tipo de retorno.

  ⚠️  NOTA IMPORTANTE:
  → Estudo observacional. Mostra ASSOCIAÇÃO, não causalidade direta.
  → Modelo ajustou por 9 covariáveis (riqueza, educação, saúde etc.).
  → Efeito mediado por escore cognitivo — o cérebro que lê envelhece diferente.


In [4]:
# --- VISUALIZAÇÃO ---

cor_leitor = '#2a8a82'
cor_nao_leitor = '#c0392b'
cor_ouro = '#c8a84b'
cor_neutra = '#9c9b94'

# ── GRÁFICO 1 — Mortalidade em 12 anos ──
fig1, ax1 = plt.subplots(figsize=(11, 6.5))

categorias = ['Não leitores', 'Leitores de livros']
proporcoes = [p_morte_nao_leitores*100, p_morte_leitores*100]
cores = [cor_nao_leitor, cor_leitor]

bars = ax1.bar(categorias, proporcoes, color=cores, alpha=0.88, width=0.42)
ax1.set_ylabel('Proporção que morreu ao longo de 12 anos (%)')
ax1.set_ylim(0, 45)
ax1.set_title('Mortalidade em 12 anos, por hábito de leitura\nCoorte de 3.635 adultos americanos (50+)',
              fontsize=13, pad=15)

for bar, v in zip(bars, proporcoes):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 1.2,
             f'{v:.0f}%', ha='center', fontweight='bold', fontsize=17)

# Anotação da diferença
ax1.annotate('', xy=(1, 27), xytext=(0, 33),
             arrowprops=dict(arrowstyle='->', color=cor_ouro, lw=2))
ax1.text(0.5, 0.68, 'redução de 20%\nno risco (HR=0,80)',
         ha='center', fontsize=11, color='#8a7020', fontweight='bold',
         transform=ax1.transAxes)

plt.figtext(0.5, 0.005,
            'Fonte: Bavishi, Slade & Levy (2016), Social Science & Medicine 164:44-48 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-032-grafico-01-mortalidade.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 — Efeito dose-resposta ──
fig2, ax2 = plt.subplots(figsize=(11, 6.5))

grupos = ['Não\nleitores', 'Leitura\nmoderada\n(~30min/dia)', 'Leitura\nalta\n(>3,5h/sem)']
hrs = [1.00, hr_tercil_2, hr_tercil_3]
cores2 = [cor_nao_leitor, cor_neutra, cor_leitor]

bars2 = ax2.bar(grupos, hrs, color=cores2, alpha=0.88, width=0.45)
ax2.axhline(y=1.0, color='#333', linestyle='--', linewidth=1.5, alpha=0.6)
ax2.set_ylabel('Hazard Ratio (risco relativo de morte)')
ax2.set_ylim(0, 1.2)
ax2.set_title('Efeito dose-resposta: quanto mais lê, menor o risco\nAjustado por idade, sexo, raça, escolaridade, riqueza, saúde e depressão',
              fontsize=13, pad=15)

for bar, v in zip(bars2, hrs):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.02,
             f'{v:.2f}', ha='center', fontweight='bold', fontsize=15)

ax2.text(0, 1.08, 'referência', ha='center', fontsize=9, color='#666', style='italic')

plt.figtext(0.5, 0.005,
            'Bavishi et al. (2016) — HR<1 significa menor risco de morrer no período | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-032-grafico-02-dose-resposta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 — Curvas de sobrevida ──
fig3, ax3 = plt.subplots(figsize=(11, 6.5))

# Curvas ilustrativas baseadas nos HRs reportados
anos = np.linspace(0, 12, 300)
lambda_nao_leitor = -np.log(1 - p_morte_nao_leitores) / 12   # hazard constante
lambda_leitor = lambda_nao_leitor * hr_leitores

sobrevida_nao_leitor = np.exp(-lambda_nao_leitor * anos) * 100
sobrevida_leitor = np.exp(-lambda_leitor * anos) * 100

ax3.plot(anos, sobrevida_nao_leitor, color=cor_nao_leitor, linewidth=2.8,
         label='Não leitores')
ax3.plot(anos, sobrevida_leitor, color=cor_leitor, linewidth=2.8,
         label='Leitores de livros')
ax3.fill_between(anos, sobrevida_nao_leitor, sobrevida_leitor,
                 alpha=0.12, color=cor_ouro)

# Marcar o ponto de 80% de sobrevida
ax3.axhline(y=80, color='#666', linestyle=':', linewidth=1.2)
ax3.text(0.3, 81.5, '80% de sobrevida', fontsize=9, color='#666', style='italic')

ax3.set_xlabel('Anos de acompanhamento')
ax3.set_ylabel('Sobrevida (%)')
ax3.set_ylim(60, 102)
ax3.set_xlim(0, 12)
ax3.set_title('Curvas de sobrevida ao longo de 12 anos\nA vantagem se abre e se mantém',
              fontsize=13, pad=15)
ax3.legend(frameon=False, loc='lower left', fontsize=11)

ax3.annotate(f'~{meses_vantagem_corrigido:.0f} meses\nde vantagem\n(corrigido ×0.80)',
             xy=(9, 78), xytext=(6.5, 68),
             fontsize=10, color='#8a7020', ha='center', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=cor_ouro, lw=1.5))

plt.figtext(0.5, 0.005,
            'Curvas ilustrativas geradas a partir dos HRs reportados por Bavishi et al. (2016) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-032-grafico-03-sobrevida.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")


✅ Gráfico 1 salvo!
✅ Gráfico 2 salvo!
✅ Gráfico 3 salvo!


### 💡 O Insight

**Aproximadamente 18 meses a mais de vida, com 30 minutos de livro por dia.**

Isso é um ano e meio. Um ano e meio de aniversários. De verões. De caminhadas. De conversas.

O custo? Meia hora que a maior parte das pessoas gasta rolando timeline sem se lembrar depois do que viu.

E o mais interessante do estudo não é só o número. É o mecanismo. A vantagem de sobrevida foi mediada por escore cognitivo. Ou seja: o cérebro que lê livros, com engajamento sustentado, com atenção profunda, envelhece diferente. E é esse envelhecimento diferente que aparece na tabela de sobrevida.

Livros não são jornais. Os pesquisadores testaram isso: a vantagem foi significativamente maior pra livros do que pra periódicos. O que faz a diferença parece ser o tipo de leitura que exige você entrar num mundo, ficar nele, seguir uma linha de pensamento longa. Não é ler qualquer coisa, é ler algo que te obriga a permanecer.

O efeito se mantém depois de controlar por riqueza, educação, gênero, saúde inicial e depressão. Ou seja: não é "gente rica lê mais e vive mais porque é rica". Depois de tirar a riqueza do modelo, o efeito de ler continua lá.

O que me pegou foi pensar no que isso significa como retorno por hora investida.

Trinta minutos por dia, ao longo de anos, é mais ou menos duzentas horas por ano. Se isso produz um ano e meio de vida a mais, você está trocando duzentas horas por doze mil e quinhentas horas. É um dos melhores retornos que a ciência já mediu pra um hábito simples.

E o meu pai, indiretamente, foi quem me ensinou isso. Não com o paper de Yale que ele nunca leu. Mas com a insistência dele de que aprender era uma forma de existir mais.

*Se o hábito mais simples e mais barato da sua semana pudesse te devolver um ano e meio de vida, você começaria hoje?*

---

### ⚠️ Limitações do Modelo

- O estudo é observacional, não experimental. Mostra associação, não causa direta. Os autores fizeram ajustes robustos pra reduzir confusão, mas não dá pra descartar completamente que quem lê livros tenha outras características não medidas que também prolongam a vida.
- Os padrões de leitura foram autorrelatados uma única vez, no início do estudo. Não capta mudanças ao longo dos 12 anos.
- Amostra composta por adultos americanos com 50 anos ou mais. Generalizar para outras idades ou culturas requer cautela.
- O "23 meses" é a estimativa bruta do modelo não ajustado. Aplico o fator de correção conservador do projeto (×0.80), o que dá aproximadamente 18 meses — um número mais defensável.
- O paper mede sobrevida, não qualidade de vida. Leitura pode ter outros efeitos (emocionais, de bem-estar, de conexão social via clubes de leitura) que este estudo específico não capturou.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
